# 01 · Signup Trust Model (GraphSAGE)

**FraudShield AI** · Model 1. Detects fake signups, multi-account abuse, bot signups, device farming, VPN abuse, trial farmers, and account-creation rings.

- Binary labels: `0 = legit_user`, `1 = abuse_user`.
- Output: `trust_score` (0-100), `risk_score` (0-100), `risk_level`.
- `trust_score = 100 - risk_score` is stored on `user.trust_score`.

In [ ]:
import pandas as pd

from trust_radar.config import SIGNUP_LABELS, FeatureConfig, SignupGNNConfig
from trust_radar.evaluation.evaluate_signup import evaluate_signup_gnn
from trust_radar.inference.predict_signup import SignupPredictor
from trust_radar.training.train_signup import train_signup_gnn
from trust_radar.utils.preprocessing import build_graph_data
from trust_radar.utils.synthetic import (
    synthesize_signup_dataset,
    synthesize_signup_edges,
)

## 1. Synthesize a schema-faithful signup dataset

The generator emits every feature in the FraudShield signup schema (identity, email, phone, browser, device, device-reputation, IP intelligence/reputation, geo-consistency, behavior, historical, relationship, and graph groups).

In [ ]:
cfg = FeatureConfig()
df = synthesize_signup_dataset(n=1500, seed=42)

print('labels        :', SIGNUP_LABELS)
print('rows          :', len(df))
print('abuse rate    :', round(float(df['label'].mean()), 3))
print('feature groups:', {k: len(v) for k, v in cfg.signup_feature_groups.items()})
df.head()

## 1b. Train / test split (row-level)

`build_graph_data()` (next section) builds the actual `train_mask` / `val_mask` / `test_mask` tensors used by GraphSAGE (70% / 15% / 15% by default). This cell makes that split visible up front: it performs the same stratified split at the row level so you can see split sizes and confirm the abuse rate is preserved in both the train and test portions before any graph/model code runs.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['label']
)
print('train rows:', len(train_df), ' abuse rate:', round(float(train_df['label'].mean()), 4))
print('test  rows:', len(test_df), ' abuse rate:', round(float(test_df['label'].mean()), 4))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['train', 'test'], [len(train_df), len(test_df)], color=['#4C72B0', '#DD8452'])
axes[0].set_title('Split size')
axes[0].set_ylabel('rows')

axes[1].bar(
    ['train', 'test'],
    [train_df['label'].mean(), test_df['label'].mean()],
    color=['#4C72B0', '#DD8452'],
)
axes[1].set_title('Abuse rate per split')
axes[1].set_ylabel('abuse rate')

fig.tight_layout()
plt.show()

> **Scaling note (10M+ nodes):** `train_signup_gnn()` currently trains GraphSAGE **full-batch** -- the entire graph (`x`, `edge_index`, all masks) is loaded at once and every epoch runs a forward pass over *all* nodes. This works fine at the sizes used in this notebook, but it is **not** the code path to use for a 10,000,000-node graph: that requires mini-batch neighbor sampling (e.g. PyTorch Geometric's `NeighborLoader`), which this training function does not yet implement. If/when you scale the signup dataset to millions of rows, that loader needs to be added before training will fit in memory / finish in reasonable time.

## 2. Build the shared device / IP graph

Nodes are signups (numeric features only); edges connect accounts that share a device or IP.

In [ ]:
nodes = df[cfg.signup_numeric_features]
edges = synthesize_signup_edges(len(df), avg_degree=5.0, seed=42)
data = build_graph_data(nodes, edges, labels=df['label'])
print(data)

## 3. Train the GraphSAGE Signup Trust Model

In [ ]:
model, history = train_signup_gnn(data, SignupGNNConfig(epochs=30, hidden_channels=64))
print('final validation ROC-AUC:', round(history['val_auc'][-1], 4))

## 4. Evaluate on the test split

Binary abuse metrics plus the resulting 0-100 trust-score distribution.

In [ ]:
metrics = evaluate_signup_gnn(model, data, split='test')
for key in ['roc_auc', 'pr_auc', 'precision', 'recall', 'f1',
            'mean_trust_score', 'mean_risk_score']:
    print(f'{key:18s}: {metrics[key]:.4f}')

### Test-set graphs

ROC curve and predicted-probability distribution computed on the held-out `test_mask` only (never seen during training).

In [ ]:
from sklearn.metrics import RocCurveDisplay

test_mask_np = data.test_mask.numpy()
with __import__('torch').no_grad():
    test_probs = model.predict_proba(data.x, data.edge_index).numpy()[test_mask_np]
test_true = data.y.numpy().ravel()[test_mask_np]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

RocCurveDisplay.from_predictions(test_true, test_probs, ax=axes[0])
axes[0].set_title(f"Test-set ROC (AUC={metrics['roc_auc']:.4f})")

axes[1].hist(test_probs[test_true == 0], bins=30, alpha=0.6, label='legit_user (0)', color='#4C72B0')
axes[1].hist(test_probs[test_true == 1], bins=30, alpha=0.6, label='abuse_user (1)', color='#C44E52')
axes[1].set_title('Test-set predicted abuse probability')
axes[1].set_xlabel('abuse_probability')
axes[1].legend()

fig.tight_layout()
plt.show()

## 5. Score signups -> trust_score, risk_score, risk_level, decision

In [ ]:
predictor = SignupPredictor(model_or_path=model)
result = predictor.predict_graph(data)
scored = pd.DataFrame({
    'trust_score': result['trust_score'],
    'risk_score': result['risk_score'],
    'risk_level': result['risk_level'],
    'decision': result['decision'],
})
scored.head(10)

In [ ]:
print('Decision distribution:')
print(scored['decision'].value_counts())
print()
print('Risk-level distribution:')
print(scored['risk_level'].value_counts())

### Single-node assessment

The full FraudShield output for one signup (matches the spec's `{trust_score, risk_score, risk_level}` payload plus the decision).

In [ ]:
sample = predictor.predict_single_node(node_idx=10, data=data)
{k: v for k, v in sample.items() if k != 'embedding'}

## 6. Decision logic

| Risk score | Action |
|------------|--------|
| 0-40 | `ALLOW` |
| 41-70 | `ALLOW_FLAG_REVIEW` |
| 71-94 | `ALLOW_HIGH_PRIORITY_REVIEW` |
| 95-100 | `TEMP_SUSPEND_MANUAL_REVIEW` |

> Disposable email, VPN, proxy, and Tor are **risk-increasing signals only** — they never trigger an automatic rejection. Every decision is a pure function of the final score.